# Uncertainty Analysis Plot Control for Three DQN Models


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

config = {
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 12,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "mathtext.fontset": "stix",
    "font.serif": ["Times New Roman"],
    "axes.unicode_minus": False,
}
rcParams.update(config)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300

PROJECT_ROOT = Path.cwd()
RESULTS_ROOT = PROJECT_ROOT / "uncertainty_analysis_results_three_dqn"
PLOT_OUTPUT_ROOT = RESULTS_ROOT / "plots_from_notebook"
PLOT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = {
    "reward5_gci": {
        "title": "DQN reward5 GCI",
        "design_baseline": PROJECT_ROOT / "results" / "DQN3_GI_new.npy",
        "real_baseline": PROJECT_ROOT / "results_m_real" / "DQN3_GI_new.npy",
    },
    "reward3_base": {
        "title": "DQN reward3 Base",
        "design_baseline": PROJECT_ROOT / "results" / "DQN3_GI.npy",
        "real_baseline": PROJECT_ROOT / "results_m_real" / "DQN3_GI.npy",
    },
    "reward3_gienv": {
        "title": "DQN reward3 GIenv",
        "design_baseline": PROJECT_ROOT / "results" / "DQN3_train_GI.npy",
        "real_baseline": PROJECT_ROOT / "results_m_real" / "DQN3_train_GI.npy",
    },
}

SCRIPT_TYPES = [
    "input_noise",
    "output_noise",
    "input_output_noise",
]

MODEL_KEYS = [
    "reward5_gci",
    "reward3_base",
    "reward3_gienv",
]

EVENT_LABEL_FILTER = None
DELTA_FILTER = [0.05, 0.10]

EVENT_ORDER_MAP = {
    "design_21": 0,
    "design_15": 1,
    "design_18": 2,
    "design_2": 3,
    "design_070": 0,
    "design_064": 1,
    "design_067": 2,
    "design_051": 3,
    "real_01": 4,
    "real_02": 5,
    "real_03": 6,
    "real_04": 7,
}
EVENT_TITLE_MAP = {
    "design_21": "Design Rainfall 21",
    "design_15": "Design Rainfall 15",
    "design_18": "Design Rainfall 18",
    "design_2": "Design Rainfall 2",
    "design_070": "Design Rainfall 21",
    "design_064": "Design Rainfall 15",
    "design_067": "Design Rainfall 18",
    "design_051": "Design Rainfall 2",
    "real_01": "Real Rainfall 1",
    "real_02": "Real Rainfall 2",
    "real_03": "Real Rainfall 3",
    "real_04": "Real Rainfall 4",
}

SCRIPT_TITLES = {
    "input_noise": "Input Noise",
    "output_noise": "Output Noise",
    "input_output_noise": "Input + Output Noise",
}

BOX_COLORS = ["#9ecae1", "#3182bd"]
MODEL_DISPLAY_ORDER = ["reward3_base", "reward3_gienv", "reward5_gci"]
MODEL_SHORT_LABELS = {
    "reward3_base": "DRL-Base",
    "reward3_gienv": "DRL-GIenv",
    "reward5_gci": "DRL-GCI",
}
MODEL_BASELINE_COLORS = {
    "reward3_base": "#a1d99b",
    "reward3_gienv": "#31a354",
    "reward5_gci": "#006837",
}
OVERFLOW_DISPLAY_SCALE = 1000.0
OVERFLOW_Y_LABEL = r"FC ($\times 10^3$ m$^3$)"
PCT_BASELINE_COLOR = "#000000"
BASELINE_LINESTYLE = "--"
BASELINE_LINEWIDTH = 1.2

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RESULTS_ROOT =", RESULTS_ROOT)
print("PLOT_OUTPUT_ROOT =", PLOT_OUTPUT_ROOT)


PROJECT_ROOT = D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI
RESULTS_ROOT = D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI\uncertainty_analysis_results_three_dqn
PLOT_OUTPUT_ROOT = D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI\uncertainty_analysis_results_three_dqn\plots_from_notebook


In [2]:
def load_summary(model_key: str, script_type: str) -> pd.DataFrame:
    csv_path = RESULTS_ROOT / model_key / script_type / "raw_data" / f"{script_type}_summary.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Summary file not found: {csv_path}")
    df = pd.read_csv(csv_path)
    if EVENT_LABEL_FILTER:
        df = df[df["event_label"].isin(EVENT_LABEL_FILTER)].copy()
    if DELTA_FILTER is not None:
        df = df[df["delta"].isin(DELTA_FILTER)].copy()
    if df.empty:
        raise ValueError(f"No rows left after filtering for {model_key} / {script_type}.")
    return df


def load_noiseless_baseline_records(model_key: str):
    spec = MODEL_SPECS[model_key]
    design_record = np.load(spec["design_baseline"], allow_pickle=True).tolist()
    real_record = np.load(spec["real_baseline"], allow_pickle=True).tolist()
    return design_record, real_record


def sorted_event_rows(df: pd.DataFrame):
    event_meta = df[["event_label", "event_title", "source", "rain_id"]].drop_duplicates().copy()
    event_meta["event_title"] = event_meta["event_label"].map(EVENT_TITLE_MAP).fillna(event_meta["event_title"])
    event_meta["event_order"] = event_meta["event_label"].map(EVENT_ORDER_MAP)
    missing_mask = event_meta["event_order"].isna()
    if missing_mask.any():
        fallback = event_meta.loc[missing_mask].sort_values(["source", "rain_id", "event_label"])
        fallback_orders = range(100, 100 + len(fallback))
        event_meta.loc[fallback.index, "event_order"] = list(fallback_orders)
    event_meta = event_meta.sort_values("event_order")
    return event_meta[["event_label", "event_title", "source", "rain_id"]].to_dict("records")


def get_boxplot_arrays(df: pd.DataFrame, event_label: str, value_column: str):
    data_arrays = []
    tick_labels = []
    for delta in sorted(df["delta"].unique()):
        values = df[(df["event_label"] == event_label) & (df["delta"] == delta)][value_column].tolist()
        data_arrays.append(values)
        tick_labels.append(f"{int(round(delta * 100))}%")
    return data_arrays, tick_labels


def scale_overflow_arrays_for_display(data_arrays):
    return [np.asarray(values, dtype=float) / OVERFLOW_DISPLAY_SCALE for values in data_arrays]


def scale_overflow_value_for_display(value):
    return float(value) / OVERFLOW_DISPLAY_SCALE


def resolve_baseline_payload(record: dict, rainfall_key: str):
    rainfall_data = record[rainfall_key]
    if "env_new" in rainfall_data:
        return rainfall_data["env_new"]
    if "env3" in rainfall_data:
        return rainfall_data["env3"]
    nested_candidates = [
        value for value in rainfall_data.values()
        if isinstance(value, dict) and "flooding" in value and "CSO" in value
    ]
    if len(nested_candidates) == 1:
        return nested_candidates[0]
    raise KeyError(f"Cannot resolve baseline payload for {rainfall_key}. Available keys: {list(rainfall_data.keys())}")


def get_noiseless_baseline_value(event_row, design_record, real_record):
    rainfall_key = f"rainfall{int(event_row['rain_id'])}"
    if event_row["source"] == "design":
        payload = resolve_baseline_payload(design_record, rainfall_key)
    else:
        payload = resolve_baseline_payload(real_record, rainfall_key)
    return float(payload["flooding"][-1] + payload["CSO"][-1])


def append_percentage_change(df: pd.DataFrame, model_key: str):
    design_record, real_record = load_noiseless_baseline_records(model_key)
    df = df.copy()
    baseline_values = []
    for _, row in df.iterrows():
        baseline_values.append(
            get_noiseless_baseline_value(
                {
                    "source": row["source"],
                    "rain_id": row["rain_id"],
                    "event_label": row["event_label"],
                    "event_title": row["event_title"],
                },
                design_record,
                real_record,
            )
        )
    df["baseline_total_overflow"] = baseline_values
    df["pct_change_vs_baseline"] = (
        (df["total_overflow"] - df["baseline_total_overflow"]) / df["baseline_total_overflow"] * 100.0
    )
    return df


def apply_boxplot_style(boxplot_dict):
    for patch, color in zip(boxplot_dict["boxes"], BOX_COLORS):
        patch.set(facecolor=color, alpha=0.9, edgecolor="black", linewidth=1.0)
    for median in boxplot_dict["medians"]:
        median.set(color="black", linewidth=1.2)
    for whisker in boxplot_dict["whiskers"]:
        whisker.set(color="black", linewidth=1.0)
    for cap in boxplot_dict["caps"]:
        cap.set(color="black", linewidth=1.0)


def make_uncertainty_legend_handles(include_baseline=True):
    handles = [
        Patch(facecolor=BOX_COLORS[0], edgecolor="black", label="5% noise"),
        Patch(facecolor=BOX_COLORS[1], edgecolor="black", label="10% noise"),
    ]
    if include_baseline:
        handles.append(
            Line2D(
                [0],
                [0],
                color=BASELINE_COLOR,
                linestyle=BASELINE_LINESTYLE,
                linewidth=BASELINE_LINEWIDTH,
                label="No-noise baseline",
            )
        )
    return handles


def get_aligned_event_positions(events):
    design_events = [event for event in events if event["source"] == "design"]
    real_events = [event for event in events if event["source"] == "real"]

    positions = {}
    for row, event in enumerate(design_events):
        positions[event["event_label"]] = (row, 0)
    for row, event in enumerate(real_events):
        positions[event["event_label"]] = (row, 1)
    return positions, design_events, real_events


def compute_pct_change_ylim(df: pd.DataFrame):
    lower = float(df["pct_change_vs_baseline"].min())
    upper = float(df["pct_change_vs_baseline"].max())
    bound = max(abs(lower), abs(upper))
    bound = max(bound * 1.1, 1.0)
    return -bound, bound


def load_all_summaries_for_script(script_type: str):
    summary_map = {}
    for model_key in MODEL_DISPLAY_ORDER:
        df = load_summary(model_key, script_type)
        summary_map[model_key] = append_percentage_change(df, model_key)
    return summary_map


def get_reference_events(summary_map: dict):
    reference_key = MODEL_DISPLAY_ORDER[0]
    return sorted_event_rows(summary_map[reference_key])


def build_grouped_positions():
    positions = {}
    xticks = []
    xticklabels = []
    current = 1.0
    model_gap = 0.9
    within_model_gap = 0.28
    group_gap = 0.9

    for model_key in MODEL_DISPLAY_ORDER:
        left = current
        right = current + within_model_gap
        positions[(model_key, 0.05)] = left
        positions[(model_key, 0.10)] = right
        xticks.append((left + right) / 2.0)
        xticklabels.append(MODEL_SHORT_LABELS[model_key])
        current += model_gap

    x_min = min(positions.values()) - 0.45
    x_max = max(positions.values()) + group_gap
    return positions, xticks, xticklabels, x_min, x_max


def style_single_box(box, color):
    box.set(facecolor=color, alpha=0.9, edgecolor="black", linewidth=1.0)


def compute_grouped_pct_ylim(summary_map: dict):
    all_values = []
    for df in summary_map.values():
        all_values.extend(df["pct_change_vs_baseline"].tolist())
    lower = float(np.min(all_values))
    upper = float(np.max(all_values))
    bound = max(abs(lower), abs(upper))
    bound = max(bound * 1.1, 1.0)
    return -bound, bound




def get_event_baseline_from_summary(df: pd.DataFrame, event_label: str):
    values = df[df["event_label"] == event_label]["baseline_total_overflow"].unique()
    if len(values) != 1:
        raise ValueError(f"Expected one baseline value for {event_label}, got {values}")
    return float(values[0]) / OVERFLOW_DISPLAY_SCALE


def plot_grouped_summary_grid(script_type: str):
    summary_map = load_all_summaries_for_script(script_type)
    output_dir = PLOT_OUTPUT_ROOT / "grouped_by_script"
    output_dir.mkdir(parents=True, exist_ok=True)

    events = get_reference_events(summary_map)
    positions_map, design_events, real_events = get_aligned_event_positions(events)
    n_rows = max(len(design_events), len(real_events), 1)
    fig, axes = plt.subplots(n_rows, 2, figsize=(9, 7), dpi=150, squeeze=False)
    box_positions, xticks, xticklabels, x_min, x_max = build_grouped_positions()

    legend_handles = [
        Patch(facecolor=BOX_COLORS[0], edgecolor="black", label="5% noise"),
        Patch(facecolor=BOX_COLORS[1], edgecolor="black", label="10% noise"),
    ] + [
        Line2D(
            [0],
            [0],
            color=MODEL_BASELINE_COLORS[model_key],
            linestyle=BASELINE_LINESTYLE,
            linewidth=BASELINE_LINEWIDTH,
            label=f"{MODEL_SHORT_LABELS[model_key]} baseline",
        )
        for model_key in MODEL_DISPLAY_ORDER
    ]

    for event in events:
        row, col = positions_map[event["event_label"]]
        ax = axes[row, col]

        for model_key in MODEL_DISPLAY_ORDER:
            df = summary_map[model_key]
            for delta, color in zip([0.05, 0.10], BOX_COLORS):
                values = df[(df["event_label"] == event["event_label"]) & (df["delta"] == delta)]["total_overflow"].to_numpy(dtype=float) / OVERFLOW_DISPLAY_SCALE
                bp = ax.boxplot(
                    [values],
                    positions=[box_positions[(model_key, delta)]],
                    widths=0.22,
                    patch_artist=True,
                    manage_ticks=False,
                )
                style_single_box(bp["boxes"][0], color)
                for median in bp["medians"]:
                    median.set(color="black", linewidth=1.2)
                for whisker in bp["whiskers"]:
                    whisker.set(color="black", linewidth=1.0)
                for cap in bp["caps"]:
                    cap.set(color="black", linewidth=1.0)

        for model_key in MODEL_DISPLAY_ORDER:
            baseline_value = get_event_baseline_from_summary(summary_map[model_key], event["event_label"])
            ax.axhline(
                baseline_value,
                color=MODEL_BASELINE_COLORS[model_key],
                linestyle=BASELINE_LINESTYLE,
                linewidth=BASELINE_LINEWIDTH,
                alpha=0.95,
            )

        ax.set_title(event["event_title"], fontweight="bold")
        ax.set_xlabel("Model")
        ax.set_ylabel(OVERFLOW_Y_LABEL)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels)
        ax.set_xlim(x_min, x_max)
        ax.margins(y=0.10)
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)

    for row in range(n_rows):
        for col in range(2):
            if not any(pos == (row, col) for pos in positions_map.values()):
                axes[row, col].axis("off")

    fig.legend(
        legend_handles,
        [handle.get_label() for handle in legend_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=5,
        frameon=False,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.96))

    save_path = output_dir / f"{script_type}_three_models_summary_grid.png"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", save_path)


def plot_grouped_pct_change_summary_grid(script_type: str):
    summary_map = load_all_summaries_for_script(script_type)
    output_dir = PLOT_OUTPUT_ROOT / "grouped_by_script"
    output_dir.mkdir(parents=True, exist_ok=True)

    events = get_reference_events(summary_map)
    positions_map, design_events, real_events = get_aligned_event_positions(events)
    n_rows = max(len(design_events), len(real_events), 1)
    fig, axes = plt.subplots(n_rows, 2, figsize=(9, 7), dpi=150, squeeze=False)
    box_positions, xticks, xticklabels, x_min, x_max = build_grouped_positions()
    pct_ylim = compute_grouped_pct_ylim(summary_map)

    legend_handles = [
        Patch(facecolor=BOX_COLORS[0], edgecolor="black", label="5% noise"),
        Patch(facecolor=BOX_COLORS[1], edgecolor="black", label="10% noise"),
        Line2D([0], [0], color=PCT_BASELINE_COLOR, linestyle=BASELINE_LINESTYLE, linewidth=BASELINE_LINEWIDTH, label="Baseline = 0%"),
    ]

    for event in events:
        row, col = positions_map[event["event_label"]]
        ax = axes[row, col]

        for model_key in MODEL_DISPLAY_ORDER:
            df = summary_map[model_key]
            for delta, color in zip([0.05, 0.10], BOX_COLORS):
                values = df[(df["event_label"] == event["event_label"]) & (df["delta"] == delta)]["pct_change_vs_baseline"].to_numpy(dtype=float)
                bp = ax.boxplot(
                    [values],
                    positions=[box_positions[(model_key, delta)]],
                    widths=0.22,
                    patch_artist=True,
                    manage_ticks=False,
                )
                style_single_box(bp["boxes"][0], color)
                for median in bp["medians"]:
                    median.set(color="black", linewidth=1.2)
                for whisker in bp["whiskers"]:
                    whisker.set(color="black", linewidth=1.0)
                for cap in bp["caps"]:
                    cap.set(color="black", linewidth=1.0)

        ax.axhline(0.0, color=PCT_BASELINE_COLOR, linestyle=BASELINE_LINESTYLE, linewidth=BASELINE_LINEWIDTH)
        ax.set_ylim(*pct_ylim)
        ax.set_title(event["event_title"], fontweight="bold")
        ax.set_xlabel("Model")
        ax.set_ylabel("Change (%)")
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels)
        ax.set_xlim(x_min, x_max)
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)

    for row in range(n_rows):
        for col in range(2):
            if not any(pos == (row, col) for pos in positions_map.values()):
                axes[row, col].axis("off")

    fig.legend(
        legend_handles,
        [handle.get_label() for handle in legend_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=3,
        frameon=False,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.96))

    save_path = output_dir / f"{script_type}_three_models_pct_change_summary_grid.png"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", save_path)


In [3]:
for script_type in SCRIPT_TYPES:
    print("\n" + "=" * 100)
    print("Plotting grouped figure for:", script_type)
    plot_grouped_summary_grid(script_type)
    plot_grouped_pct_change_summary_grid(script_type)

print("\nAll grouped three-model uncertainty-analysis plots have been generated.")



Plotting grouped figure for: input_noise
Saved: D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI\uncertainty_analysis_results_three_dqn\plots_from_notebook\grouped_by_script\input_noise_three_models_summary_grid.png
Saved: D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI\uncertainty_analysis_results_three_dqn\plots_from_notebook\grouped_by_script\input_noise_three_models_pct_change_summary_grid.png

Plotting grouped figure for: output_noise
Saved: D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI\uncertainty_analysis_results_three_dqn\plots_from_notebook\grouped_by_script\output_noise_three_models_summary_grid.png
Saved: D:\from_E\School\School\FYP\Green_infrastructure_RTC\New_reward\v9_new_reward_nonnegative\DRL-GI\uncertainty_analysis_results_three_dqn\plots_from_notebook\grouped_by_script\output_noise_three_models_pct_change_summary_grid.png